# Compare the learned extractor with rules

Demonstration documents only. These notebooks inspect the committed evidence; they do not claim a separate full-dataset training run.

In [1]:
from pathlib import Path
import json
import pandas as pd
root = Path.cwd()
e = json.loads((root / 'results/manifest.json').read_text(encoding='utf-8'))
print('Population:', e['scope'])


Population: {'generated': {'train': 48, 'validation': 24, 'test': 24}, 'cord': {'train': 800, 'validation': 100, 'test': 100}, 'limitations': ['Full original CORD 800/100/100 splits are used, with a project-specific subtotal/tax/total and line-item mapping, not the official benchmark metric.', 'Generated invoices share four templates; performance does not establish generalization to unseen vendors or layouts.', 'Small test populations give wide intervals; the chosen costs are illustrative (review 1, wrong approval 20).', 'CORD gold words cover annotated semantic lines; full-page OCR CER/WER includes unannotated text.', 'Gold/real OCR input gaps include token segmentation and layout conventions, so they are not pure additive character-recognition loss.', 'Rules are conservative on receipts without invoice date, vendor and PO evidence, so those receipts go to review.', 'Other-class classification uses generated text tokens; no real miscellaneous mailroom documents were evaluated.', 'Extr

In [2]:
rows = pd.DataFrame([{'source':r['source'], 'backend':r['backend'], 'input':r['ocr_engine'],
                     'field_f1':r['field_f1']['value'], 'interval':r['field_f1'].get('ci')}
                    for r in e['extraction']])
print(rows.to_string(index=False))
print('Recorded retraining:', e.get('retrain'))
assert rows['backend'].nunique() >= 2

   source backend     input  field_f1                                  interval
generated   rules      gold  1.000000                                [1.0, 1.0]
generated   rules tesseract  1.000000                                [1.0, 1.0]
generated  tokens      gold  1.000000                                [1.0, 1.0]
generated  tokens tesseract  1.000000                                [1.0, 1.0]
     cord   rules      gold  0.635069             [0.5522482638888889, 0.70625]
     cord   rules tesseract  0.248264           [0.1784375, 0.3199739583333333]
     cord  tokens      gold  0.854514  [0.7989583333333333, 0.9040190972222223]
     cord  tokens tesseract  0.310069 [0.22871527777777778, 0.3906423611111111]
Recorded retraining: {'before': {'value': 0.4480555555555556, 'ci': [0.3641111111111111, 0.5371458333333332], 'n': 120, 'method': 'document bootstrap, 500 replicates, seed 11027, 95% percentile'}, 'after': {'value': 0.44388888888888894, 'ci': [0.3584513888888889, 0.530875], 'n': 

## Rejected shortcut / negative control

In [3]:
# Rejected shortcut: choosing the largest test estimate is not a promotion test.
leader = rows.sort_values('field_f1', ascending=False).iloc[0]
print('Largest point estimate:', leader.to_dict())
print('This observation alone has no paired significance or deployment guarantee.')

Largest point estimate: {'source': 'generated', 'backend': 'rules', 'input': 'gold', 'field_f1': 1.0, 'interval': [1.0, 1.0]}
This observation alone has no paired significance or deployment guarantee.
